In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)

In [2]:
df = pd.read_excel(r"dataset/HousePricePrediction.xlsx")

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (2919, 13)


,Id,MSSubClass,MSZoning,LotArea,LotConfig,BldgType,OverallCond,YearBuilt,YearRemodAdd,Exterior1st,BsmtFinSF2,TotalBsmtSF,SalePrice
0,0,60,RL,8450,Inside,1Fam,5,2003,2003,VinylSd,0.0,856.0,208500.0
1,1,20,RL,9600,FR2,1Fam,8,1976,1976,MetalSd,0.0,1262.0,181500.0
2,2,60,RL,11250,Inside,1Fam,5,2001,2002,VinylSd,0.0,920.0,223500.0
3,3,70,RL,9550,Corner,1Fam,5,1915,1970,Wd Sdng,0.0,756.0,140000.0
4,4,60,RL,14260,FR2,1Fam,5,2000,2000,VinylSd,0.0,1145.0,250000.0


In [3]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 2919 entries, 0 to 2918
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Id            2919 non-null   int64  
 1   MSSubClass    2919 non-null   int64  
 2   MSZoning      2915 non-null   str    
 3   LotArea       2919 non-null   int64  
 4   LotConfig     2919 non-null   str    
 5   BldgType      2919 non-null   str    
 6   OverallCond   2919 non-null   int64  
 7   YearBuilt     2919 non-null   int64  
 8   YearRemodAdd  2919 non-null   int64  
 9   Exterior1st   2918 non-null   str    
 10  BsmtFinSF2    2918 non-null   float64
 11  TotalBsmtSF   2918 non-null   float64
 12  SalePrice     1460 non-null   float64
dtypes: float64(3), int64(6), str(4)
memory usage: 352.3 KB
None


In [4]:
print(df.isnull().sum())

Id                 0
MSSubClass         0
MSZoning           4
LotArea            0
LotConfig          0
BldgType           0
OverallCond        0
YearBuilt          0
YearRemodAdd       0
Exterior1st        1
BsmtFinSF2         1
TotalBsmtSF        1
SalePrice       1459
dtype: int64


In [5]:
features = [
    "MSSubClass",
    "LotArea",
    "OverallCond",
    "YearBuilt",
    "YearRemodAdd",
    "BsmtFinSF2",
    "TotalBsmtSF",
    "MSZoning",
    "LotConfig",
    "BldgType",
    "Exterior1st"
]

target = "SalePrice"

In [6]:
X = df[features].copy()

y = df[target].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X.head()

X shape: (2919, 11)
y shape: (2919,)


,MSSubClass,LotArea,OverallCond,YearBuilt,YearRemodAdd,BsmtFinSF2,TotalBsmtSF,MSZoning,LotConfig,BldgType,Exterior1st
0,60,8450,5,2003,2003,0.0,856.0,RL,Inside,1Fam,VinylSd
1,20,9600,8,1976,1976,0.0,1262.0,RL,FR2,1Fam,MetalSd
2,60,11250,5,2001,2002,0.0,920.0,RL,Inside,1Fam,VinylSd
3,70,9550,5,1915,1970,0.0,756.0,RL,Corner,1Fam,Wd Sdng
4,60,14260,5,2000,2000,0.0,1145.0,RL,FR2,1Fam,VinylSd


In [7]:
numerical_cols = [
    "MSSubClass",
    "LotArea",
    "OverallCond",
    "YearBuilt",
    "YearRemodAdd",
    "BsmtFinSF2",
    "TotalBsmtSF"
]

In [8]:
categorical_cols = [
    "MSZoning",
    "LotConfig",
    "BldgType",
    "Exterior1st"
]

In [9]:
for col in numerical_cols:
    X[col] = X[col].fillna(X[col].median())

for col in categorical_cols:
    X[col] = X[col].fillna(X[col].mode()[0])

In [10]:
print(X.isnull().sum())

MSSubClass      0
LotArea         0
OverallCond     0
YearBuilt       0
YearRemodAdd    0
BsmtFinSF2      0
TotalBsmtSF     0
MSZoning        0
LotConfig       0
BldgType        0
Exterior1st     0
dtype: int64


In [11]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Validation data:", X_valid.shape)

Training data: (2335, 11)
Validation data: (584, 11)


In [12]:

# Numerical → StandardScaler
# Categorical → OneHotEncoder
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_cols
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_cols
        )
    ]
)

In [13]:
# create svr_pipeline
svr_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "svr",
            SVR(
                kernel="rbf",
                C=100,
                epsilon=0.05,
                gamma="scale"
            )
        )
    ]
)

In [14]:
# Target transformation
model_SVR = TransformedTargetRegressor(
    regressor=svr_pipeline,
    transformer=StandardScaler()
)

In [15]:
print("Missing SalePrice values:", df["SalePrice"].isna().sum())

Missing SalePrice values: 1459


In [16]:
# Remove rows where SalePrice is missing
df = df.dropna(subset=["SalePrice"]).copy()

print("Missing SalePrice values:", df["SalePrice"].isna().sum())

Missing SalePrice values: 0


In [17]:
features = [
    "MSSubClass",
    "LotArea",
    "OverallCond",
    "YearBuilt",
    "YearRemodAdd",
    "BsmtFinSF2",
    "TotalBsmtSF",
    "MSZoning",
    "LotConfig",
    "BldgType",
    "Exterior1st"
]

X = df[features].copy()
y = df["SalePrice"].copy()

In [18]:
print("X missing values:")
print(X.isna().sum())

print("\ny missing values:")
print(y.isna().sum())

X missing values:
MSSubClass      0
LotArea         0
OverallCond     0
YearBuilt       0
YearRemodAdd    0
BsmtFinSF2      0
TotalBsmtSF     0
MSZoning        0
LotConfig       0
BldgType        0
Exterior1st     0
dtype: int64

y missing values:
0


In [19]:
numerical_cols = [
    "MSSubClass",
    "LotArea",
    "OverallCond",
    "YearBuilt",
    "YearRemodAdd",
    "BsmtFinSF2",
    "TotalBsmtSF"
]

categorical_cols = [
    "MSZoning",
    "LotConfig",
    "BldgType",
    "Exterior1st"
]

for col in numerical_cols:
    X[col] = X[col].fillna(X[col].median())

for col in categorical_cols:
    X[col] = X[col].fillna(X[col].mode()[0])

In [20]:
print(X.isna().sum().sum())
print(y.isna().sum())

0
0


In [21]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [22]:
#Train the model
model_SVR.fit(
    X_train,
    y_train
)

print("Model training completed successfully!")

Model training completed successfully!


In [23]:
# Prediction
y_pred = model_SVR.predict(X_valid)

print(y_pred[:10])

[152395.02298785 275378.58348531  58388.80350653 133528.11000946
 292658.06813235 140287.51700535 193204.86286478 148270.89291071
 132321.43875233 176372.4349734 ]


In [24]:
# Evaluate model
mae = mean_absolute_error(
    y_valid,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_valid,
        y_pred
    )
)

r2 = r2_score(
    y_valid,
    y_pred
)

mape = mean_absolute_percentage_error(
    y_valid,
    y_pred
)

In [25]:
y_pred = model_SVR.predict(X_valid)

print(y_pred[:10])

[152395.02298785 275378.58348531  58388.80350653 133528.11000946
 292658.06813235 140287.51700535 193204.86286478 148270.89291071
 132321.43875233 176372.4349734 ]


In [26]:
print("Model Performance")
print("-------------------------")
print("MAE  :", mae)
print("RMSE :", rmse)
print("R²   :", r2)
print("MAPE :", mape)
print("MAPE %:", mape * 100)

Model Performance
-------------------------
MAE  : 27994.284336496898
RMSE : 43863.15279395213
R²   : 0.7491663169132471
MAPE : 0.17399276590204188
MAPE %: 17.399276590204188


In [27]:
comparison = pd.DataFrame({
    "Actual Price": y_valid.values,
    "Predicted Price": y_pred
})

comparison.head(10)

,Actual Price,Predicted Price
0,154500.0,152395.022988
1,325000.0,275378.583485
2,115000.0,58388.803507
3,159000.0,133528.110009
4,315500.0,292658.068132
5,75500.0,140287.517005
6,311500.0,193204.862865
7,146000.0,148270.892911
8,84500.0,132321.438752
9,135500.0,176372.434973


In [29]:
joblib.dump(
    model_SVR,
    "model_SVT.pkl"
)

print("model_SVR.pkl saved successfully!")

model_SVR.pkl saved successfully!


In [1]:
pip freeze > requirements.txt


Note: you may need to restart the kernel to use updated packages.
